# YOLO26m-P2 Training for Pickleball Ball Detection
**Before running:** Go to Runtime → Change runtime type → Select **T4 GPU**

In [ ]:
# Step 1: Install ultralytics
!pip install -q ultralytics

In [ ]:
# Step 2: Check GPU
import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB')

In [ ]:
# Step 3: Upload your dataset zip
# Upload pickleball-dataset.zip (created from training/pickleball.v2i.yolo26/ folder)
from google.colab import files
uploaded = files.upload()

In [ ]:
# Step 4: Unzip dataset
!unzip -q pickleball-dataset.zip -d dataset/

In [ ]:
# Step 5: Create data.yaml with correct paths
import yaml
import os

# Auto-detect dataset structure
base = '/content/dataset'
# Handle nested folder (if zip contains a subfolder)
contents = os.listdir(base)
if 'train' not in contents and len(contents) == 1:
    base = os.path.join(base, contents[0])

data_config = {
    'train': os.path.join(base, 'train/images'),
    'val': os.path.join(base, 'valid/images'),
    'test': os.path.join(base, 'test/images'),
    'nc': 1,
    'names': ['ball']
}

with open('/content/data.yaml', 'w') as f:
    yaml.dump(data_config, f)

print('data.yaml created:')
print(yaml.dump(data_config))
print(f'Train images: {len(os.listdir(data_config["train"]))}')
print(f'Valid images: {len(os.listdir(data_config["val"]))}')

In [ ]:
# Step 6: Train YOLO26m-P2
from ultralytics import YOLO

# Load P2 architecture with pretrained yolo26m weights
model = YOLO('yolo26m-p2.yaml').load('yolo26m.pt')

# Train
model.train(
    data='/content/data.yaml',
    imgsz=1280,             # High res — T4 has 15GB VRAM
    device='0',
    batch=8,
    epochs=100,
    workers=2,
    name='pickleball-yolo26m-p2',
    patience=25,
    lr0=0.005,
    fliplr=0.5,
    degrees=0.0,
    mosaic=1.0,
    copy_paste=0.3,
    hsv_v=0.4,
    resume=False,
)

In [ ]:
# Step 7: Download the best weights
from google.colab import files
files.download('runs/detect/pickleball-yolo26m-p2/weights/best.pt')

In [ ]:
# Step 8 (Optional): View training results
from IPython.display import Image, display
display(Image('runs/detect/pickleball-yolo26m-p2/results.png', width=800))